Installation et téléchargement données

In [2]:
!pip install wbgapi
import wbgapi as wb

import s3fs

fs = s3fs.S3FileSystem(client_kwargs={"endpoint_url": "https://minio.lab.sspcloud.fr"})

MY_BUCKET = "eleanorpaineau"
print(fs.ls(MY_BUCKET))


[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: pip install --upgrade pip
['eleanorpaineau/diffusion']


In [4]:
countries = ["AUS", "AUT", "BEL", "CAN", "CHL", "COL", "CRI", "CZE", "DNK", "EST", "FIN",
    "FRA", "DEU", "GRC", "HUN", "ISL", "IRL", "ISR", "ITA", "JPN", "KOR",
    "LVA", "LTU", "LUX", "MEX", "NLD", "NZL", "NOR", "POL", "PRT", "SVK",
    "SVN", "ESP", "SWE", "CHE", "TUR", "GBR", "USA"]
taux_chomage = wb.data.DataFrame("SL.UEM.TOTL.ZS", economy=countries , time=range(1991, 2024))
pib_hab_ppa = wb.data.DataFrame("NY.GDP.PCAP.PP.KD", economy=countries, time=range(1991, 2024))



with fs.open(f"{MY_BUCKET}/diffusion/projet/raw/wb_chomage.csv", "w") as f:
    taux_chomage.to_csv(f)
with fs.open(f"{MY_BUCKET}/diffusion/projet/raw/wb_pib_hab_ppa.csv", "w") as f:
    pib_hab_ppa.to_csv(f)


taux_chomage=taux_chomage.reset_index(names="economy")
pib_hab_ppa=pib_hab_ppa.reset_index(names="economy")
print(pib_hab_ppa.tail(10))


   economy        YR1991        YR1992        YR1993        YR1994  \
28     NLD  43833.478505  44245.519825  44490.750447  45532.798666   
29     NOR  57949.698676  59669.759020  61019.816103  63730.896464   
30     NZL  29014.149942  29029.581136  30532.588017  31669.540651   
31     POL  11879.753925  12141.235132  12563.104536  13200.161416   
32     PRT  28443.794536  28776.047577  28153.615518  28348.865839   
33     SVK  14955.954208  13946.163079  14157.181437  14976.577355   
34     SVN  23078.992950  21850.057102  22524.959188  23752.437725   
35     SWE  39469.071647  38783.639768  37762.540822  39000.827723   
36     TUR  12057.935436  12451.045120  13174.467116  12343.004185   
37     USA  43747.630735  44664.873586  45292.738068  46543.319920   

          YR1995        YR1996        YR1997        YR1998        YR1999  ...  \
28  46720.281932  48095.986838  49881.894752  51879.359671  54134.351454  ...   
29  66033.883783  68964.975622  72084.813919  73610.189669  74703.5

Mise en forme - taux de chômage

In [5]:
import pandas as pd

# On transforme notre tableau afin que les colonnes des années deviennent des valeurs de la colonne "year"
df_taux_chomage=taux_chomage.melt(
    id_vars=["economy"],
    var_name="year",
    value_name="Taux chômage"
)

#Tri de notre tableau par ordre alphabétique des codes des pays 
df_taux_chomage=df_taux_chomage.sort_values(by="economy", ascending = True)

#Création d'un tableau qui associe les codes pays à leurs noms complets en anglais
noms = pd.DataFrame({
    "economy": ["AUS", "AUT", "BEL", "CAN", "CHL", "COL", "CRI", "CZE", "DNK", "EST", "FIN",
    "FRA", "DEU", "GRC", "HUN", "ISL", "IRL", "ISR", "ITA", "JPN", "KOR",
    "LVA", "LTU", "LUX", "MEX", "NLD", "NZL", "NOR", "POL", "PRT", "SVK",
    "SVN", "ESP", "SWE", "CHE", "TUR", "GBR", "USA"],

    "pays": ["Australia", "Austria", "Belgium", "Canada", "Chile", "Colombia", "Costa Rica",
    "Czechia", "Denmark", "Estonia", "Finland", "France", "Germany", "Greece",
    "Hungary", "Iceland", "Ireland", "Israel", "Italy", "Japan", "South Korea",
    "Latvia", "Lithuania", "Luxembourg", "Mexico", "Netherlands", "New Zealand",
    "Norway", "Poland", "Portugal", "Slovakia", "Slovenia", "Spain", "Sweden",
    "Switzerland", "Turkey", "United Kingdom", "United States"]
})
#On joint notre tableau pour ajouter les noms des pays, en regroupant par le code pays
df_taux_chomage=df_taux_chomage.merge(noms, on="economy", how="left")
print(df_taux_chomage.head(10))

  economy    year  Taux chômage       pays
0     AUS  YR1991         9.586  Australia
1     AUS  YR1992        10.733  Australia
2     AUS  YR1993        10.879  Australia
3     AUS  YR1994         9.724  Australia
4     AUS  YR1995         8.473  Australia
5     AUS  YR1996         8.509  Australia
6     AUS  YR1997         8.367  Australia
7     AUS  YR1998         7.684  Australia
8     AUS  YR1999         6.876  Australia
9     AUS  YR2000         6.288  Australia


Mise en forme - pib/tête en ppa

In [6]:
# De la même manière, on transforme notre tableau afin que les années soient une variable de la colonne "year"
df_pib_hab_ppa= pib_hab_ppa.melt(
    id_vars=["economy"],
    var_name="year",
    value_name="PIB/tête PPA"
)

#Tri du tableau par ordre alphabétique des codes pays
df_pib_hab_ppa=df_pib_hab_ppa.sort_values(by="economy", ascending = True)

#Merge des tableaux avec taux de chômage et pib/hab ppa en faisant la jointure par pays et année
df_eco=df_taux_chomage.merge(df_pib_hab_ppa, on=["economy", "year"], how="right")

#Pour que le tableau soit plus propre, on réorganise simplement les colonnes
df_eco = df_eco[["economy", "pays", "year", "Taux chômage", "PIB/tête PPA"]]

#Transformation de la colonne "year" pour n'avoir que l'année et non "YRAAAA" (ex : "YR2004")
df_eco["year"]=df_eco["year"].str.replace("YR","")
print(df_eco.tail(10))


     economy           pays  year  Taux chômage  PIB/tête PPA
1244     USA  United States  2014         6.168  63026.877919
1245     USA  United States  2015         5.280  64367.718891
1246     USA  United States  2016         4.869  65025.985598
1247     USA  United States  2017         4.355  66164.032001
1248     USA  United States  2018         3.896  67728.442492
1249     USA  United States  2019         3.669  69121.458002
1250     USA  United States  2020         8.055  67406.831133
1251     USA  United States  2021         5.349  71441.231981
1252     USA  United States  2022         3.650  72828.743091
1253     USA  United States  2023         3.638  74351.671820
